In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [3]:
BASE = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "requirements.txt").is_file()
)


In [4]:
CLEAN = BASE / "data" / "clean"
FIG = BASE / "reports" / "figures"

FIG.mkdir(parents=True, exist_ok=True)


In [9]:
table1 = pd.read_csv(CLEAN / "table1_province.csv")
table2 = pd.read_csv(CLEAN / "table2_month_age_sex.csv")
table3 = pd.read_csv(CLEAN / "table3_top_problems.csv")

In [16]:
ont = table1[table1["province"] == "Ont."].iloc[0]
can = table1[table1["province"] == "Total"].iloc[0]
print(ont.to_string())
print(can.to_string())

province                    Ont.
total_visits           6606941.0
ctas_1_3_discharged    3925489.0
ctas_4_5_discharged    1266822.0
admitted                769932.0
median_ctas_1_3              3.7
median_ctas_4_5              2.3
median_admitted             13.5
p90_ctas_1_3                 8.2
p90_ctas_4_5                 5.7
p90_admitted                38.3
province                    Total
total_visits           15628870.0
ctas_1_3_discharged     7850030.0
ctas_4_5_discharged     4144772.0
admitted                1797938.0
median_ctas_1_3               4.1
median_ctas_4_5               2.7
median_admitted              15.5
p90_ctas_1_3                 10.2
p90_ctas_4_5                  7.8
p90_admitted                 46.5


In [17]:
print("===1. Ontario vs Canada: 2025-2026 ===")
print(f"Ontario share of Canada visits: {ont['total_visits']}/ {can['total_visits']*100:.1f} %")

print(f"Ontario admitted rate: {ont['admitted']/ont['total_visits']*100:.1f}% | Canada: {can['admitted']/can['total_visits']*100:.1f}%")
print(f"Ontario CTAS I-III discharged share of Ontario visits: {ont['ctas_1_3_discharged']/ont['total_visits']*100:.1f}%")
print(f"Ontario CTAS IV-V discharged share: {ont['ctas_4_5_discharged']/ont['total_visits']*100:.1f}%")
for col in ["median_ctas_1_3", "median_ctas_4_5", "median_admitted", "p90_ctas_1_3", "p90_ctas_4_5", "p90_admitted"]:
    print(f"{col}: Ontario {ont[col]}h, Canada {can[col]}h, difference {ont[col]-can[col]:+.1f}h")

print("\n=== Provinces ranked by admitted median LOS ===")
ranked = table1[table1["province"] != "Total"].sort_values("median_admitted")
print(ranked[["province", "total_visits", "median_admitted", "p90_admitted"]].to_string(index=False))



===1. Ontario vs Canada: 2025-2026 ===
Ontario share of Canada visits: 6606941.0/ 1562887000.0 %
Ontario admitted rate: 11.7% | Canada: 11.5%
Ontario CTAS I-III discharged share of Ontario visits: 59.4%
Ontario CTAS IV-V discharged share: 19.2%
median_ctas_1_3: Ontario 3.7h, Canada 4.1h, difference -0.4h
median_ctas_4_5: Ontario 2.3h, Canada 2.7h, difference -0.4h
median_admitted: Ontario 13.5h, Canada 15.5h, difference -2.0h
p90_ctas_1_3: Ontario 8.2h, Canada 10.2h, difference -2.0h
p90_ctas_4_5: Ontario 5.7h, Canada 7.8h, difference -2.1h
p90_admitted: Ontario 38.3h, Canada 46.5h, difference -8.2h

=== Provinces ranked by admitted median LOS ===
province  total_visits  median_admitted  p90_admitted
 Sask.**      702894.0              9.7          36.8
    Y.T.       39046.0             10.6          26.8
   Alta.     2328090.0             13.2          42.3
    Ont.     6606941.0             13.5          38.3
  Man.**      463666.0             17.3          53.6
    Que.     3609912

# Chart 1: admitted median and p90 by province

In [30]:
prov = ranked.copy()
fig, ax = plt.subplots(figsize=(9, 5))

In [31]:
ax.barh(prov["province"], prov["median_admitted"], label="Median admitted LOS (h)")
ax.barh(prov["province"], prov["p90_admitted"] - prov["median_admitted"],
        left=prov["median_admitted"], alpha=0.35, label="Up to 90th percentile (h)")
ax.set_xlabel("Hours in ED")
ax.set_title("Admitted patients: median and 90th percentile ED stay by province, 2025-2026")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "admitted_los_by_province.png", dpi=150)
plt.close(fig)

# Ontario monthly pattern (All sexes)

In [20]:
print("\n=== 2. Ontario monthly pattern (All sexes) ===")
months = ["April 2025", "May 2025", "June 2025", "July 2025", "August 2025",
          "September 2025", "October 2025", "November 2025", "December 2025",
          "January 2026", "February 2026", "March 2026"]



=== 2. Ontario monthly pattern (All sexes) ===


In [21]:
ont_m = table2[(table2["province"] == "Ont.") & (table2["sex"] == "All")].set_index("month").loc[months]
ont_m = ont_m.reset_index()


In [22]:
print(ont_m[["month", "age_all"]].to_string(index=False))
print(f"Peak month: {ont_m.loc[ont_m['age_all'].idxmax(), 'month']} ({ont_m['age_all'].max():,.0f})")
print(f"Lowest month: {ont_m.loc[ont_m['age_all'].idxmin(), 'month']} ({ont_m['age_all'].min():,.0f})")
print(f"Monthly average: {ont_m['age_all'].mean():,.0f}")

         month  age_all
    April 2025 541444.0
      May 2025 567559.0
     June 2025 554945.0
     July 2025 577112.0
   August 2025 568005.0
September 2025 557974.0
  October 2025 559879.0
 November 2025 536973.0
 December 2025 590597.0
  January 2026 518249.0
 February 2026 485239.0
    March 2026 548965.0
Peak month: December 2025 (590,597)
Lowest month: February 2026 (485,239)
Monthly average: 550,578


In [23]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ont_m["month"], ont_m["age_all"], marker="o")
ax.set_ylabel("ED visits")
ax.set_title("Ontario ED visits by month, April 2025 to March 2026")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG / "ontario_monthly_visits.png", dpi=150)
plt.close(fig)

In [ ]:
print("\n=== 3. Ontario age and sex mix, full year (All months) ===")
ont_all = table2[(table2["province"] == "Ont.") & (table2["sex"] == "All")]



=== 3. Ontario age and sex mix, full year (All months) ===
Age 0-4: 429,505 (6.5%)
Age 5-19: 801,408 (12.1%)
Age 20-64: 3,602,467 (54.5%)
Age 65+: 1,772,584 (26.8%)
Female: 3,469,879 (52.5%)
Male: 3,135,842 (47.5%)


In [ ]:
for col, label in [("age_0_4", "0-4"), ("age_5_19", "5-19"), ("age_20_64", "20-64"), ("age_65_plus", "65+")]:
    total = ont_all[col].sum()
    print(f"Age {label}: {total:,.0f} ({total/ont_all['age_all'].sum()*100:.1f}%)")


In [ ]:
ont_sex = table2[(table2["province"] == "Ont.") & (table2["sex"].isin(["Female", "Male"]))]
sex_tot = ont_sex.groupby("sex")["age_all"].sum()
print(f"Female: {sex_tot['Female']:,.0f} ({sex_tot['Female']/sex_tot.sum()*100:.1f}%)")
print(f"Male: {sex_tot['Male']:,.0f} ({sex_tot['Male']/sex_tot.sum()*100:.1f}%)")

In [25]:
print("\n=== 4. Top 10 main problems ===")
print(f"Top 10 total visits: {table3['visits'].sum():,.0f} ({table3['visits'].sum()/can['total_visits']*100:.1f}% of all Canada visits)")
print(table3.sort_values("visits", ascending=False).to_string(index=False))
print(f"Highest admitted share: {table3.loc[table3['admitted_pct'].idxmax(), 'main_problem']} ({table3['admitted_pct'].max()}%)")
print(f"Longest 90th percentile stay: {table3.loc[table3['p90_los_hours'].idxmax(), 'main_problem']} ({table3['p90_los_hours'].max()}h)")



=== 4. Top 10 main problems ===
Top 10 total visits: 2,132,884 (13.6% of all Canada visits)
                                                           main_problem  visits  p90_los_hours  non_admitted_pct  admitted_pct
                                           1) Abdominal and pelvic pain  454357           10.5              96.4           3.6
                                            2) Pain in throat and chest  381489            9.0              97.4           2.6
                                               3) Dorsalgia (back pain)  223838            9.0              97.0           3.0
                                   4) Other disorders of urinary system  177674           12.7              88.1          11.9
5) Acute upper respiratory infections of multiple and unspecified sites  170394            6.0              98.7           1.3
                                        6) Open wound of wrist and hand  155800            5.7              99.7           0.3
                  

In [26]:
fig, ax = plt.subplots(figsize=(9, 5))
top = table3.sort_values("visits")
ax.barh(top["main_problem"].str[:45], top["visits"])
ax.set_xlabel("ED visits")
ax.set_title("Top 10 main problems in the ED, Canada, 2025-2026")
fig.tight_layout()
fig.savefig(FIG / "top10_main_problems.png", dpi=150)
plt.close(fig)

In [27]:
for yr, fname in [("2024-2025", "nacrs-ed-2024-2025.xlsx"), ("2025-2026", "nacrs-ed-2025-2026.xlsx")]:
    raw = pd.read_excel(BASE / "data" / "raw" / fname, sheet_name="Table 1", header=1, nrows=10)
    raw.columns = list(table1.columns)
    o = raw[raw["province"] == "Ont."].iloc[0]
    c = raw[raw["province"] == "Total"].iloc[0]
    print(f"{yr}: Canada {c['total_visits']:,.0f}, Ontario {o['total_visits']:,.0f}, "
          f"Canada admitted median {c['median_admitted']}h p90 {c['p90_admitted']}h, "
          f"Ontario admitted median {o['median_admitted']}h p90 {o['p90_admitted']}h")
print("Canada change: 15,628,870 - 16,102,787 = -473,917 (-2.9%)")
print("Ontario change: 6,606,941 - 6,616,431 = -9,490 (-0.1%)")
print("Figures saved to", FIG)


2024-2025: Canada 16,102,787, Ontario 6,616,431, Canada admitted median 16.0h p90 48.5h, Ontario admitted median 14.4h p90 44.4h
2025-2026: Canada 15,628,870, Ontario 6,606,941, Canada admitted median 15.5h p90 46.5h, Ontario admitted median 13.5h p90 38.3h
Canada change: 15,628,870 - 16,102,787 = -473,917 (-2.9%)
Ontario change: 6,606,941 - 6,616,431 = -9,490 (-0.1%)
Figures saved to /Users/mac/Documents/DataAnalysis/learningProjects/dataproject/reports/figures
